In [1]:
# =========================
# Google Colab / Python
# Batch EMG filtering + feature extraction
# Based on your previous logic:
# 5 Hz high-pass -> rectification -> 10 Hz low-pass
# 100 ms window + 50% overlap
# Activity == "Lifting-Lowering"
# Features: MAV, VAR, MAD, WL, TP, SM1, SM2, SM3
# =========================

# Optional: mount Google Drive in Colab
# from google.colab import drive
# drive.mount('/content/drive')

import os
import re
import glob
import json
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt


# =========================
# CONFIG
# =========================
FS = 500
WINDOW_MS = 100
STEP_MS = 50

RAW_DIR = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\Raw_40"
FILTERED_DIR = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\Filtered_40"
FEATURES_DIR = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\Features_40"

MERGED_FEATURES_CSV = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\Merged_Features_40.csv"
PROCESS_LOG_CSV = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\emg_processing_log_40.csv"
SUMMARY_JSON = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\emg_processing_summary_40.json"

TARGET_ACTIVITY = "Lifting-Lowering"

# If your files use different metadata column names, edit here
META_COLS_CANDIDATES = [
    "Participant",
    "Sex",
    "Age",
    "Activity",
    "Lifting Height",
    "Box Mass",
    "Load Type",
    "Lifting Depth",
    "Timestamp",
]



# Preferred EMG column patterns
EMG_MVC_PATTERN = r"^EMG_MVC_CH\d+$"
EMG_RAW_PATTERN = r"^EMG_CH\d+$"

os.makedirs(FILTERED_DIR, exist_ok=True)
os.makedirs(FEATURES_DIR, exist_ok=True)


# =========================
# FILTER FUNCTIONS
# =========================
def _safe_filtfilt(b, a, x):
    x = np.asarray(x, dtype=float)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    if len(x) < max(len(a), len(b)) * 3:
        return x.copy()

    try:
        return filtfilt(b, a, x)
    except Exception:
        return x.copy()


def highpass_emg(emg, fs, cutoff=5, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype="highpass")
    return _safe_filtfilt(b, a, emg)


def lowpass_emg(emg, fs, cutoff=10, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype="lowpass")
    return _safe_filtfilt(b, a, emg)


def process_emg_channel(emg_raw, fs):
    emg_raw = np.asarray(emg_raw, dtype=float)
    emg_raw = np.nan_to_num(emg_raw, nan=0.0, posinf=0.0, neginf=0.0)
    emg_hp = highpass_emg(emg_raw, fs, cutoff=5, order=4)
    emg_rect = np.abs(emg_hp)
    emg_env = lowpass_emg(emg_rect, fs, cutoff=10, order=4)
    return emg_env


# =========================
# FEATURE FUNCTIONS
# =========================
def td_fd_features(x, fs):
    x = np.asarray(x, dtype=float)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    n = len(x)
    if n < 2:
        return {
            "MAV": 0.0,
            "VAR": 0.0,
            "MAD": 0.0,
            "WL": 0.0,
            "TP": 0.0,
            "SM1": 0.0,
            "SM2": 0.0,
            "SM3": 0.0,
        }

    mav = np.mean(np.abs(x))
    var = np.var(x)
    mad = np.mean(np.abs(x - np.mean(x)))
    wl = np.sum(np.abs(np.diff(x)))

    X = np.fft.rfft(x)
    freqs = np.fft.rfftfreq(n, d=1.0 / fs)
    P = np.abs(X) ** 2
    total_power = np.sum(P)

    if total_power <= 0:
        sm1 = sm2 = sm3 = 0.0
    else:
        sm1 = np.sum(freqs * P) / total_power
        sm2 = np.sum((freqs ** 2) * P) / total_power
        sm3 = np.sum((freqs ** 3) * P) / total_power

    return {
        "MAV": float(mav),
        "VAR": float(var),
        "MAD": float(mad),
        "WL": float(wl),
        "TP": float(total_power),
        "SM1": float(sm1),
        "SM2": float(sm2),
        "SM3": float(sm3),
    }


# =========================
# HELPERS
# =========================
def find_csv_files(root_dir):
    return sorted(glob.glob(os.path.join(root_dir, "**", "*.csv"), recursive=True))


def infer_emg_cols(df):
    mvc_cols = [c for c in df.columns if re.match(EMG_MVC_PATTERN, str(c))]
    raw_cols = [c for c in df.columns if re.match(EMG_RAW_PATTERN, str(c))]

    def sort_key(col):
        m = re.search(r"(\d+)$", str(col))
        return int(m.group(1)) if m else 999

    mvc_cols = sorted(mvc_cols, key=sort_key)
    raw_cols = sorted(raw_cols, key=sort_key)

    if len(mvc_cols) >= 1:
        return mvc_cols
    if len(raw_cols) >= 1:
        return raw_cols

    # fallback: columns containing "EMG"
    fallback = [c for c in df.columns if "EMG" in str(c).upper()]
    fallback = sorted(fallback, key=sort_key)
    return fallback


def get_existing_meta_cols(df):
    return [c for c in META_COLS_CANDIDATES if c in df.columns]


def get_trial_col(df):
    if "Timestamp" in df.columns:
        return "Timestamp"
    return None


def get_activity_col(df):
    if "Activity" in df.columns:
        return "Activity"
    return None


def get_timestamp_value(row):
    return row["Timestamp"] if "Timestamp" in row.index else np.nan


def contiguous_true_segments(mask):
    start = None
    for i, flag in enumerate(mask):
        if flag and start is None:
            start = i
        elif not flag and start is not None:
            yield start, i - 1
            start = None
    if start is not None:
        yield start, len(mask) - 1


def participant_from_filename(path):
    stem = Path(path).stem
    m = re.search(r"(P\d+)", stem, flags=re.IGNORECASE)
    return m.group(1) if m else stem


# =========================
# ONE FILE: FILTER
# =========================
def filter_one_file(input_csv, output_csv, fs=FS):
    df = pd.read_csv(input_csv)
    emg_cols = infer_emg_cols(df)

    if len(emg_cols) == 0:
        raise ValueError(f"No EMG columns found in {input_csv}")

    for col in emg_cols:
        df[col] = process_emg_channel(df[col].values, fs)

    df.to_csv(output_csv, index=False)

    return {
        "n_rows": int(len(df)),
        "emg_cols": emg_cols,
        "meta_cols": get_existing_meta_cols(df),
    }


# =========================
# ONE FILE: FEATURE EXTRACTION
# =========================
def extract_features_one_file(filtered_csv, output_csv, fs=FS, window_ms=WINDOW_MS, step_ms=STEP_MS):
    df = pd.read_csv(filtered_csv)

    emg_cols = infer_emg_cols(df)
    if len(emg_cols) == 0:
        raise ValueError(f"No EMG columns found in {filtered_csv}")

    meta_cols = get_existing_meta_cols(df)
    trial_col = get_trial_col(df)
    activity_col = get_activity_col(df)

    if trial_col is None:
        raise ValueError(f"'Timeline' column not found in {filtered_csv}")
    if activity_col is None:
        raise ValueError(f"'Activity' column not found in {filtered_csv}")

    window_samples = int(fs * window_ms / 1000)
    step_samples = int(fs * step_ms / 1000)

    rows = []

    for trial_name, trial_df in df.groupby(trial_col, sort=False):
        trial_df = trial_df.reset_index(drop=True)

        is_lift = trial_df[activity_col].astype(str).eq(TARGET_ACTIVITY).values

        for seg_start, seg_end in contiguous_true_segments(is_lift):
            seg = trial_df.iloc[seg_start:seg_end + 1].reset_index(drop=True)
            seg_len = len(seg)

            if seg_len < window_samples:
                continue

            for w_start in range(0, seg_len - window_samples + 1, step_samples):
                w_end = w_start + window_samples
                w = seg.iloc[w_start:w_end]
                center_idx = w_start + window_samples // 2
                center_row = seg.iloc[center_idx]

                feat = {}

                for col in meta_cols:
                    feat[col] = center_row[col]

                if "Participant" not in feat:
                    feat["Participant"] = participant_from_filename(filtered_csv)

                feat["Source_File"] = Path(filtered_csv).name
                feat["Segment_Start_Index"] = int(seg_start)
                feat["Segment_End_Index"] = int(seg_end)
                feat["Window_Start_Offset"] = int(w_start)
                feat["Window_End_Offset"] = int(w_end - 1)
                feat["Window_Center_Timestamp"] = get_timestamp_value(center_row)

                for ch in emg_cols:
                    x = w[ch].values.astype(float)
                    f = td_fd_features(x, fs)
                    for name, val in f.items():
                        feat[f"{ch}_{name}"] = val

                rows.append(feat)

    features_df = pd.DataFrame(rows)
    features_df.to_csv(output_csv, index=False)

    return {
        "n_feature_rows": int(len(features_df)),
        "emg_cols": emg_cols,
        "meta_cols": meta_cols,
    }


# =========================
# BATCH PIPELINE
# =========================
def run_batch_pipeline(raw_dir=RAW_DIR, filtered_dir=FILTERED_DIR, features_dir=FEATURES_DIR):
    csv_files = find_csv_files(raw_dir)
    if len(csv_files) == 0:
        raise FileNotFoundError(f"No CSV files found under: {raw_dir}")

    logs = []
    feature_tables = []

    for i, input_csv in enumerate(csv_files, start=1):
        base_name = Path(input_csv).stem
        filtered_csv = os.path.join(filtered_dir, f"{base_name}_butter_low_high.csv")
        features_csv = os.path.join(features_dir, f"{base_name}_features.csv")

        log_row = {
            "input_csv": input_csv,
            "filtered_csv": filtered_csv,
            "features_csv": features_csv,
            "status": "started",
            "filter_ok": False,
            "feature_ok": False,
            "n_raw_rows": np.nan,
            "n_feature_rows": np.nan,
            "n_emg_cols": np.nan,
            "error": "",
        }

        try:
            filter_info = filter_one_file(input_csv, filtered_csv, fs=FS)
            log_row["filter_ok"] = True
            log_row["n_raw_rows"] = filter_info["n_rows"]
            log_row["n_emg_cols"] = len(filter_info["emg_cols"])

            feature_info = extract_features_one_file(
                filtered_csv,
                features_csv,
                fs=FS,
                window_ms=WINDOW_MS,
                step_ms=STEP_MS,
            )
            log_row["feature_ok"] = True
            log_row["n_feature_rows"] = feature_info["n_feature_rows"]
            log_row["status"] = "success"

            if os.path.exists(features_csv):
                df_feat = pd.read_csv(features_csv)
                if len(df_feat) > 0:
                    feature_tables.append(df_feat)

        except Exception as e:
            log_row["status"] = "failed"
            log_row["error"] = f"{type(e).__name__}: {str(e)}\n{traceback.format_exc()}"

        logs.append(log_row)
        print(f"[{i}/{len(csv_files)}] {Path(input_csv).name} -> {log_row['status']}")

    log_df = pd.DataFrame(logs)
    log_df.to_csv(PROCESS_LOG_CSV, index=False)

    if len(feature_tables) > 0:
        merged_df = pd.concat(feature_tables, ignore_index=True, sort=False)
        merged_df.to_csv(MERGED_FEATURES_CSV, index=False)
    else:
        merged_df = pd.DataFrame()

    summary = {
        "raw_dir": raw_dir,
        "filtered_dir": filtered_dir,
        "features_dir": features_dir,
        "n_input_files": int(len(csv_files)),
        "n_success_files": int((log_df["status"] == "success").sum()),
        "n_failed_files": int((log_df["status"] == "failed").sum()),
        "merged_feature_rows": int(len(merged_df)),
        "merged_feature_cols": int(merged_df.shape[1]) if len(merged_df) > 0 else 0,
        "participants_in_merged": sorted(map(str, merged_df["Participant"].dropna().unique().tolist())) if "Participant" in merged_df.columns else [],
        "sex_counts": merged_df["Sex"].value_counts(dropna=False).to_dict() if "Sex" in merged_df.columns else {},
        "weight_counts": merged_df["Box Mass"].value_counts(dropna=False).to_dict() if "Box Mass" in merged_df.columns else {},
        "load_type_counts": merged_df["Load Type"].value_counts(dropna=False).to_dict() if "Load Type" in merged_df.columns else {},
    }

    with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    print("\nDone.")
    print(f"Log saved to: {PROCESS_LOG_CSV}")
    print(f"Merged features saved to: {MERGED_FEATURES_CSV}")
    print(f"Summary saved to: {SUMMARY_JSON}")

    return log_df, merged_df, summary


# =========================
# RUN
# =========================
log_df, merged_features_df, summary = run_batch_pipeline()

# quick checks
print("\n=== Processing log head ===")
print(log_df.head())

print("\n=== Merged features shape ===")
print(merged_features_df.shape)

if len(merged_features_df) > 0:
    print("\n=== Columns sample ===")
    print(merged_features_df.columns[:30].tolist())

    if "Participant" in merged_features_df.columns:
        print("\n=== Participant counts ===")
        print(merged_features_df["Participant"].value_counts())

    if "Sex" in merged_features_df.columns:
        print("\n=== Sex counts ===")
        print(merged_features_df["Sex"].value_counts(dropna=False))

    if "Box Mass" in merged_features_df.columns:
        print("\n=== Box Mass counts ===")
        print(merged_features_df["Box Mass"].value_counts(dropna=False).sort_index())

    if "Load Type" in merged_features_df.columns:
        print("\n=== Load Type counts ===")
        print(merged_features_df["Load Type"].value_counts(dropna=False))


[1/40] P10_EMG.csv -> failed
[2/40] P11_EMG.csv -> failed
[3/40] P12_EMG.csv -> failed
[4/40] P13_EMG.csv -> failed
[5/40] P14_EMG.csv -> failed
[6/40] P15_EMG.csv -> failed
[7/40] P16_EMG.csv -> failed
[8/40] P17_EMG.csv -> failed
[9/40] P18_EMG.csv -> failed
[10/40] P19_EMG.csv -> failed
[11/40] P1_EMG.csv -> failed
[12/40] P20_EMG.csv -> failed
[13/40] P21_EMG.csv -> failed
[14/40] P22_EMG.csv -> failed
[15/40] P23_EMG.csv -> failed
[16/40] P24_EMG.csv -> failed
[17/40] P25_EMG.csv -> failed
[18/40] P26_EMG.csv -> failed
[19/40] P27_EMG.csv -> failed
[20/40] P28_EMG.csv -> failed
[21/40] P29_EMG.csv -> failed
[22/40] P2_EMG.csv -> failed
[23/40] P30_EMG.csv -> failed
[24/40] P31_EMG.csv -> failed
[25/40] P32_EMG.csv -> failed
[26/40] P33_EMG.csv -> failed
[27/40] P34_EMG.csv -> failed
[28/40] P35_EMG.csv -> failed
[29/40] P36_EMG.csv -> failed
[30/40] P37_EMG.csv -> failed
[31/40] P38_EMG.csv -> failed
[32/40] P39_EMG.csv -> failed
[33/40] P3_EMG.csv -> failed
[34/40] P40_EMG.csv ->

In [ ]:
import os
import json
import traceback
from pathlib import Path

import numpy as np
import pandas as pd

# =========================
# PATHS / SETTINGS
# =========================
FILTERED_DIR = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\Filtered"
FEATURES_DIR = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\Features"
MERGED_FEATURES_CSV = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\Merged_Features.csv"

FS = 500
WINDOW_MS = 100
STEP_MS = 50
TARGET_ACTIVITY = "Lifting-Lowering"   # None if you want all activities

WINDOW_SAMPLES = int(FS * WINDOW_MS / 1000)
STEP_SAMPLES = int(FS * STEP_MS / 1000)

META_COLS_CANDIDATES = [
    "Timestamp",
    "Participant",
    "Sex",
    "Age",
    "Stature",
    "Body Mass",
    "Trial Number",
    "Activity",
    "Lifting Height",
    "Lowering Height",
    "Box Mass",
    "Load Type",
    "Lifting Depth",
]

TIME_COL_CANDIDATES = ["Timeline", "Timestamp", "Time", "time", "timestamp"]

Path(FEATURES_DIR).mkdir(parents=True, exist_ok=True)


# =========================
# HELPERS
# =========================
def find_time_col(df):
    for c in TIME_COL_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"No time column found. Columns: {df.columns.tolist()}")


def get_emg_cols(df):
    emg_cols = [c for c in df.columns if c.startswith("EMG_CH")]
    if not emg_cols:
        raise ValueError("No EMG_CH* columns found.")
    return emg_cols


def zero_crossings(x, threshold=1e-8):
    x = np.asarray(x, dtype=float)
    x1 = x[:-1]
    x2 = x[1:]
    return np.sum(((x1 * x2) < 0) & (np.abs(x1 - x2) >= threshold))


def slope_sign_changes(x, threshold=1e-8):
    x = np.asarray(x, dtype=float)
    d1 = x[1:-1] - x[:-2]
    d2 = x[1:-1] - x[2:]
    return np.sum(((d1 * d2) > 0) & ((np.abs(d1) >= threshold) | (np.abs(d2) >= threshold)))


def waveform_length(x):
    x = np.asarray(x, dtype=float)
    return np.sum(np.abs(np.diff(x)))


def compute_emg_features(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return {
            "mean": np.nan,
            "std": np.nan,
            "min": np.nan,
            "max": np.nan,
            "mav": np.nan,
            "rms": np.nan,
            "iemg": np.nan,
            "wl": np.nan,
            "zc": np.nan,
            "ssc": np.nan,
        }

    return {
        "mean": np.mean(x),
        "std": np.std(x, ddof=0),
        "min": np.min(x),
        "max": np.max(x),
        "mav": np.mean(np.abs(x)),
        "rms": np.sqrt(np.mean(x ** 2)),
        "iemg": np.sum(np.abs(x)),
        "wl": waveform_length(x),
        "zc": zero_crossings(x),
        "ssc": slope_sign_changes(x),
    }


def make_group_cols(df, time_col, emg_cols):
    exclude = set(emg_cols + [time_col])
    group_cols = [c for c in META_COLS_CANDIDATES if c in df.columns and c not in exclude]
    return group_cols


# =========================
# FEATURE EXTRACTION
# =========================
def extract_features_one_file(filtered_csv, out_dir=FEATURES_DIR):
    df = pd.read_csv(filtered_csv)

    time_col = find_time_col(df)
    emg_cols = get_emg_cols(df)

    if TARGET_ACTIVITY is not None and "Activity" in df.columns:
        df = df[df["Activity"] == TARGET_ACTIVITY].copy()

    if df.empty:
        raise ValueError("No rows left after activity filtering.")

    df = df.sort_values(time_col).reset_index(drop=True)
    group_cols = make_group_cols(df, time_col, emg_cols)

    if group_cols:
        grouped = df.groupby(group_cols, dropna=False, sort=False)
    else:
        grouped = [(("ALL",), df)]

    all_rows = []

    for _, g in grouped:
        g = g.sort_values(time_col).reset_index(drop=True)

        if len(g) < WINDOW_SAMPLES:
            continue

        for start in range(0, len(g) - WINDOW_SAMPLES + 1, STEP_SAMPLES):
            end = start + WINDOW_SAMPLES
            w = g.iloc[start:end]

            row = {}

            for c in group_cols:
                row[c] = w.iloc[0][c]

            row["window_start_idx"] = start
            row["window_end_idx"] = end - 1
            row["window_size"] = WINDOW_SAMPLES
            row["step_size"] = STEP_SAMPLES
            row["start_time"] = w.iloc[0][time_col]
            row["end_time"] = w.iloc[-1][time_col]

            for ch in emg_cols:
                feats = compute_emg_features(w[ch].values)
                for feat_name, feat_val in feats.items():
                    row[f"{ch}_{feat_name}"] = feat_val

            all_rows.append(row)

    if not all_rows:
        raise ValueError("No valid windows extracted.")

    feat_df = pd.DataFrame(all_rows)

    in_name = Path(filtered_csv).stem
    out_fp = Path(out_dir) / f"{in_name}_features.csv"
    feat_df.to_csv(out_fp, index=False)

    return feat_df, str(out_fp)


def run_feature_batch(filtered_dir=FILTERED_DIR, out_dir=FEATURES_DIR):
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    csv_files = sorted(Path(filtered_dir).glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found under: {filtered_dir}")

    logs = []

    for i, fp in enumerate(csv_files, 1):
        try:
            feat_df, out_fp = extract_features_one_file(str(fp), out_dir=out_dir)
            logs.append({
                "input_file": str(fp),
                "output_file": out_fp,
                "status": "success",
                "n_rows": len(feat_df),
                "error_type": "",
                "error_message": ""
            })
            print(f"[{i}/{len(csv_files)}] {fp.name} -> success ({len(feat_df)} rows)")
        except Exception as e:
            logs.append({
                "input_file": str(fp),
                "output_file": "",
                "status": "failed",
                "n_rows": 0,
                "error_type": type(e).__name__,
                "error_message": str(e)
            })
            print(f"[{i}/{len(csv_files)}] {fp.name} -> failed: {type(e).__name__}: {e}")

    log_df = pd.DataFrame(logs)
    log_fp = Path(out_dir) / "feature_extraction_log.csv"
    log_df.to_csv(log_fp, index=False)

    return log_df


# =========================
# MERGE
# =========================
def run_merge_features(features_dir=FEATURES_DIR, merged_csv=MERGED_FEATURES_CSV):
    feature_files = sorted(
        [fp for fp in Path(features_dir).glob("*_features.csv") if fp.name != Path(merged_csv).name]
    )
    if not feature_files:
        raise FileNotFoundError(f"No feature CSV files found under: {features_dir}")

    dfs = []
    for fp in feature_files:
        df = pd.read_csv(fp)
        df["source_file"] = fp.name
        dfs.append(df)

    merged_df = pd.concat(dfs, ignore_index=True)
    merged_df.to_csv(merged_csv, index=False)

    summary = {
        "n_feature_files": len(feature_files),
        "n_rows_merged": int(len(merged_df)),
        "n_cols_merged": int(merged_df.shape[1]),
        "merged_csv": merged_csv,
    }

    with open(str(Path(features_dir) / "merge_summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(f"Merged -> {merged_csv}")
    print(summary)

    return merged_df, summary


# =========================
# RUN
# =========================
feature_log_df = run_feature_batch()
merged_features_df, merge_summary = run_merge_features()


[1/2] P12_EMG_butter_low_high.csv -> success (15769 rows)


KeyboardInterrupt: 

In [13]:
FEATURES_DIR = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\Features"
MERGED_FEATURES_CSV = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\Merged_Features\Merged_Features.csv"

merged_features_df, merge_summary = run_merge_features(
    features_dir=FEATURES_DIR,
    merged_csv=MERGED_FEATURES_CSV
)


Merged -> C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\Merged_Features\Merged_Features.csv
{'n_feature_files': 39, 'n_rows_merged': 801315, 'n_cols_merged': 99, 'merged_csv': 'C:\\Users\\lilin\\OneDrive\\Desktop\\ECE5424\\Capstone_Project\\EMG\\Merged_Features\\Merged_Features.csv'}


In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import welch


# =========================
# CONFIG
# =========================
FILTERED_DIR = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\Filtered_40"

OUT_ROOT = r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\TimeFreq_40"
FEATURES_DIR = str(Path(OUT_ROOT) / "Features")
MERGED_FEATURES_CSV = str(Path(OUT_ROOT) / "Merged_Features" / "Merged_Features_TimeFreq_40.csv")
FEATURE_LOG_CSV = str(Path(OUT_ROOT) / "feature_extraction_log.csv")
MERGE_SUMMARY_JSON = str(Path(OUT_ROOT) / "merge_summary.json")

FS = 500
WINDOW_MS = 200
STEP_MS = 100
TARGET_ACTIVITY = "Lifting-Lowering"   # set to None if you want all activities

WINDOW_SAMPLES = int(FS * WINDOW_MS / 1000)
STEP_SAMPLES = int(FS * STEP_MS / 1000)

TIME_COL_CANDIDATES = ["Timeline", "Timestamp", "Time", "time", "timestamp"]

META_COLS_CANDIDATES = [
    "Timestamp",
    "Timeline",
    "Participant",
    "Sex",
    "Age",
    "Stature",
    "Body Mass",
    "Trial Number",
    "Activity",
    "Lifting Height",
    "Lowering Height",
    "Box Mass",
    "Box Weight",
    "Load Type",
    "Lifting Depth",
]

Path(FEATURES_DIR).mkdir(parents=True, exist_ok=True)
Path(MERGED_FEATURES_CSV).parent.mkdir(parents=True, exist_ok=True)


# =========================
# HELPERS
# =========================
def find_time_col(df):
    for c in TIME_COL_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"No time column found. Columns: {df.columns.tolist()}")


def get_emg_cols(df):
    emg_cols = [c for c in df.columns if c.startswith("EMG_CH")]
    if not emg_cols:
        raise ValueError("No EMG_CH* columns found.")
    return sorted(emg_cols)


def zero_crossings(x, threshold=1e-8):
    x = np.asarray(x, dtype=float)
    x1 = x[:-1]
    x2 = x[1:]
    return int(np.sum(((x1 * x2) < 0) & (np.abs(x1 - x2) >= threshold)))


def slope_sign_changes(x, threshold=1e-8):
    x = np.asarray(x, dtype=float)
    d1 = x[1:-1] - x[:-2]
    d2 = x[1:-1] - x[2:]
    return int(np.sum(((d1 * d2) > 0) & ((np.abs(d1) >= threshold) | (np.abs(d2) >= threshold))))


def waveform_length(x):
    x = np.asarray(x, dtype=float)
    return float(np.sum(np.abs(np.diff(x))))


def spectral_entropy(psd):
    psd = np.asarray(psd, dtype=float)
    psd_sum = np.sum(psd)
    if psd_sum <= 0:
        return np.nan
    p = psd / psd_sum
    p = p[p > 0]
    if len(p) == 0:
        return np.nan
    return float(-(np.sum(p * np.log2(p))) / np.log2(len(psd)))


def median_frequency(freqs, psd):
    freqs = np.asarray(freqs, dtype=float)
    psd = np.asarray(psd, dtype=float)
    total_power = np.trapz(psd, freqs)
    if total_power <= 0:
        return np.nan
    cumulative = np.cumsum((psd[:-1] + psd[1:]) / 2 * np.diff(freqs))
    idx = np.searchsorted(cumulative, total_power / 2)
    idx = min(idx, len(freqs) - 1)
    return float(freqs[idx])


def band_power(freqs, psd, low, high):
    freqs = np.asarray(freqs, dtype=float)
    psd = np.asarray(psd, dtype=float)
    mask = (freqs >= low) & (freqs < high)
    if not np.any(mask):
        return np.nan
    return float(np.trapz(psd[mask], freqs[mask]))


def compute_time_features(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]

    if len(x) == 0:
        return {
            "mean": np.nan,
            "std": np.nan,
            "median": np.nan,
            "iqr": np.nan,
            "min": np.nan,
            "max": np.nan,
            "mav": np.nan,
            "rms": np.nan,
            "iemg": np.nan,
            "wl": np.nan,
            "zc": np.nan,
            "ssc": np.nan,
        }

    q25 = np.percentile(x, 25)
    q75 = np.percentile(x, 75)

    return {
        "mean": float(np.mean(x)),
        "std": float(np.std(x, ddof=0)),
        "median": float(np.median(x)),
        "iqr": float(q75 - q25),
        "min": float(np.min(x)),
        "max": float(np.max(x)),
        "mav": float(np.mean(np.abs(x))),
        "rms": float(np.sqrt(np.mean(x ** 2))),
        "iemg": float(np.sum(np.abs(x))),
        "wl": waveform_length(x),
        "zc": zero_crossings(x),
        "ssc": slope_sign_changes(x),
    }


def compute_frequency_features(x, fs):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]

    if len(x) < 8:
        return {
            "mnf": np.nan,
            "mdf": np.nan,
            "peak_freq": np.nan,
            "total_power": np.nan,
            "spec_entropy": np.nan,
            "bp_20_60": np.nan,
            "bp_60_120": np.nan,
            "bp_120_200": np.nan,
        }

    freqs, psd = welch(
        x,
        fs=fs,
        window="hann",
        nperseg=min(len(x), 128),
        noverlap=0,
        detrend="constant",
        scaling="density",
    )

    if len(freqs) == 0 or np.all(~np.isfinite(psd)):
        return {
            "mnf": np.nan,
            "mdf": np.nan,
            "peak_freq": np.nan,
            "total_power": np.nan,
            "spec_entropy": np.nan,
            "bp_20_60": np.nan,
            "bp_60_120": np.nan,
            "bp_120_200": np.nan,
        }

    psd = np.nan_to_num(psd, nan=0.0, posinf=0.0, neginf=0.0)
    total_power = float(np.trapz(psd, freqs))

    if total_power <= 0:
        mnf = np.nan
        mdf = np.nan
        peak_freq = np.nan
        spec_ent = np.nan
    else:
        mnf = float(np.sum(freqs * psd) / np.sum(psd))
        mdf = median_frequency(freqs, psd)
        peak_freq = float(freqs[np.argmax(psd)])
        spec_ent = spectral_entropy(psd)

    return {
        "mnf": mnf,
        "mdf": mdf,
        "peak_freq": peak_freq,
        "total_power": total_power,
        "spec_entropy": spec_ent,
        "bp_20_60": band_power(freqs, psd, 20, 60),
        "bp_60_120": band_power(freqs, psd, 60, 120),
        "bp_120_200": band_power(freqs, psd, 120, 200),
    }


def compute_emg_features(x, fs):
    feats = {}
    feats.update(compute_time_features(x))
    feats.update(compute_frequency_features(x, fs))
    return feats


def make_group_cols(df, time_col, emg_cols):
    exclude = set(emg_cols + [time_col])
    group_cols = [c for c in META_COLS_CANDIDATES if c in df.columns and c not in exclude]
    return group_cols


# =========================
# FEATURE EXTRACTION
# =========================
def extract_features_one_file(filtered_csv, out_dir=FEATURES_DIR):
    df = pd.read_csv(filtered_csv)

    time_col = find_time_col(df)
    emg_cols = get_emg_cols(df)

    if TARGET_ACTIVITY is not None and "Activity" in df.columns:
        df = df[df["Activity"] == TARGET_ACTIVITY].copy()

    if df.empty:
        raise ValueError("No rows left after activity filtering.")

    df = df.sort_values(time_col).reset_index(drop=True)
    group_cols = make_group_cols(df, time_col, emg_cols)

    if group_cols:
        grouped = df.groupby(group_cols, dropna=False, sort=False)
    else:
        grouped = [(("ALL",), df)]

    all_rows = []

    for _, g in grouped:
        g = g.sort_values(time_col).reset_index(drop=True)

        if len(g) < WINDOW_SAMPLES:
            continue

        for start in range(0, len(g) - WINDOW_SAMPLES + 1, STEP_SAMPLES):
            end = start + WINDOW_SAMPLES
            w = g.iloc[start:end]

            row = {}

            for c in group_cols:
                row[c] = w.iloc[0][c]

            row["window_start_idx"] = int(start)
            row["window_end_idx"] = int(end - 1)
            row["window_size"] = int(WINDOW_SAMPLES)
            row["step_size"] = int(STEP_SAMPLES)
            row["start_time"] = w.iloc[0][time_col]
            row["end_time"] = w.iloc[-1][time_col]

            for ch in emg_cols:
                feats = compute_emg_features(w[ch].values, FS)
                for feat_name, feat_val in feats.items():
                    row[f"{ch}_{feat_name}"] = feat_val

            all_rows.append(row)

    if not all_rows:
        raise ValueError("No valid windows extracted.")

    feat_df = pd.DataFrame(all_rows)

    in_name = Path(filtered_csv).stem
    out_fp = Path(out_dir) / f"{in_name}_timefreq_features.csv"
    feat_df.to_csv(out_fp, index=False)

    return feat_df, str(out_fp)


def run_feature_batch(filtered_dir=FILTERED_DIR, out_dir=FEATURES_DIR):
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    csv_files = sorted(Path(filtered_dir).glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found under: {filtered_dir}")

    logs = []

    for i, fp in enumerate(csv_files, 1):
        try:
            feat_df, out_fp = extract_features_one_file(str(fp), out_dir=out_dir)
            logs.append({
                "input_file": str(fp),
                "output_file": out_fp,
                "status": "success",
                "n_rows": int(len(feat_df)),
                "error_type": "",
                "error_message": ""
            })
            print(f"[{i}/{len(csv_files)}] {fp.name} -> success ({len(feat_df)} rows)")
        except Exception as e:
            logs.append({
                "input_file": str(fp),
                "output_file": "",
                "status": "failed",
                "n_rows": 0,
                "error_type": type(e).__name__,
                "error_message": str(e)
            })
            print(f"[{i}/{len(csv_files)}] {fp.name} -> failed: {type(e).__name__}: {e}")

    log_df = pd.DataFrame(logs)
    log_df.to_csv(FEATURE_LOG_CSV, index=False)

    return log_df


# =========================
# MERGE
# =========================
def run_merge_features(features_dir=FEATURES_DIR, merged_csv=MERGED_FEATURES_CSV):
    Path(merged_csv).parent.mkdir(parents=True, exist_ok=True)

    feature_files = sorted(Path(features_dir).glob("*_timefreq_features.csv"))
    if not feature_files:
        raise FileNotFoundError(f"No feature CSV files found under: {features_dir}")

    dfs = []
    skipped = []

    for fp in feature_files:
        if fp.stat().st_size == 0:
            skipped.append((fp.name, "empty file"))
            continue

        try:
            df = pd.read_csv(fp)
            if df.empty or len(df.columns) == 0:
                skipped.append((fp.name, "no rows/columns"))
                continue
            df["source_file"] = fp.name
            dfs.append(df)
        except Exception as e:
            skipped.append((fp.name, f"{type(e).__name__}: {e}"))

    if not dfs:
        raise ValueError("No valid feature CSV files available for merging.")

    merged_df = pd.concat(dfs, ignore_index=True)
    merged_df.to_csv(merged_csv, index=False)

    summary = {
        "filtered_dir": FILTERED_DIR,
        "features_dir": features_dir,
        "merged_csv": merged_csv,
        "fs": FS,
        "window_ms": WINDOW_MS,
        "step_ms": STEP_MS,
        "window_samples": WINDOW_SAMPLES,
        "step_samples": STEP_SAMPLES,
        "target_activity": TARGET_ACTIVITY,
        "n_feature_files_found": len(feature_files),
        "n_feature_files_merged": len(dfs),
        "n_feature_files_skipped": len(skipped),
        "skipped_files": skipped,
        "n_rows_merged": int(len(merged_df)),
        "n_cols_merged": int(merged_df.shape[1]),
    }

    with open(MERGE_SUMMARY_JSON, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(f"Merged -> {merged_csv}")
    print(summary)

    return merged_df, summary


# =========================
# RUN
# =========================
feature_log_df = run_feature_batch()
merged_features_df, merge_summary = run_merge_features()

[1/40] P10_EMG_butter_low_high.csv -> success (8744 rows)
[2/40] P11_EMG_butter_low_high.csv -> success (12382 rows)
[3/40] P12_EMG_butter_low_high.csv -> success (7802 rows)
[4/40] P13_EMG_butter_low_high.csv -> success (11203 rows)
[5/40] P14_EMG_butter_low_high.csv -> success (9104 rows)
[6/40] P15_EMG_butter_low_high.csv -> success (9670 rows)
[7/40] P16_EMG_butter_low_high.csv -> success (9618 rows)
[8/40] P17_EMG_butter_low_high.csv -> success (7975 rows)
[9/40] P18_EMG_butter_low_high.csv -> success (14544 rows)
[10/40] P19_EMG_butter_low_high.csv -> success (8951 rows)
[11/40] P1_EMG_butter_low_high.csv -> success (12058 rows)
[12/40] P20_EMG_butter_low_high.csv -> success (8305 rows)
[13/40] P21_EMG_butter_low_high.csv -> success (9068 rows)
[14/40] P22_EMG_butter_low_high.csv -> success (9110 rows)
[15/40] P23_EMG_butter_low_high.csv -> success (10060 rows)
[16/40] P24_EMG_butter_low_high.csv -> success (12101 rows)
[17/40] P25_EMG_butter_low_high.csv -> success (10527 rows)
